In [ ]:
import pandas as pd
import seaborn as sns
sns.set_style("whitegrid")
from collections import Counter
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import sklearn
import scanpy as sc
from sklearn.model_selection import train_test_split
plt.style.use('ggplot')
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "DejaVu Sans",
})
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('retina')
try:
    import hnswlib
    hnswlib_imported = True
except ImportError:
    hnswlib_imported = False
    print("hnswlib not installed! We highly recommend installing it for fast similarity search.")
    print("To install it, run: pip install hnswlib")
import os, random
import numpy as np
import torch
import torch.backends.cudnn as cudnn

Reproducibility

In [ ]:
def fix_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    cudnn.deterministic = True
    cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(1)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

fix_seed(42)

Load Dataset

In [ ]:
# sampled_adata = sc.read_h5ad("data/human_spleen_updated.h5ad")
# sampled_adata = sc.read_h5ad("data/gse155468.h5ad")
sampled_adata = sc.read_h5ad("data/GSE194122_cite_BMMC_processed.h5ad")

dataset = "BMMC"
datatype = "sc"

train_adata = sc.read_h5ad(f"data/cell_type_annotation/{dataset}/{dataset}_train.h5ad")
test_adata = sc.read_h5ad(f"data/cell_type_annotation/{dataset}/{dataset}_test.h5ad")

In [ ]:
with open("data/GPT_3_5_gene_embeddings.pickle", "rb") as fp:
    GPT_3_5_gene_embeddings = pickle.load(fp)

embed_genes = list(GPT_3_5_gene_embeddings.keys())
common_genes = np.intersect1d(
    np.intersect1d(train_adata.var.index, test_adata.var.index),
    embed_genes
)
print(f"Using {len(common_genes)} genes shared across train/test and embeddings")

In [ ]:
train_adata.var_names_make_unique()
test_adata.var_names_make_unique()
train_adata = train_adata[:, common_genes]
test_adata  = test_adata[:, common_genes]

embed_df = pd.DataFrame.from_dict(GPT_3_5_gene_embeddings, orient="index")
embed_df = embed_df.reindex(common_genes).fillna(0.0)
lookup_embed = embed_df.values.astype(np.float32)

In [ ]:
train_mat = train_adata.X.toarray() if hasattr(train_adata.X, "toarray") else train_adata.X
test_mat  = test_adata.X.toarray() if hasattr(test_adata.X, "toarray") else test_adata.X

genePT_w_embed_train = train_mat @ lookup_embed / len(common_genes)
genePT_w_embed_test  = test_mat  @ lookup_embed / len(common_genes)

In [ ]:
y_train = train_adata.obs["cell_type"].astype(str).values
y_test  = test_adata.obs["cell_type"].astype(str).values

print(f"Train cells: {len(y_train)}, Test cells: {len(y_test)}")
print(f"Unique train types: {len(np.unique(y_train))}, Unique test types: {len(np.unique(y_test))}")

In [ ]:
ref_cell_embeddings = genePT_w_embed_train
test_embed = genePT_w_embed_test

kNN Classification

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, auc
from sklearn.preprocessing import label_binarize
# cell type clustering
# very quick test
k = 7  # number of neighbors
neighbors_list_gpt_v1 = []
if hnswlib_imported:
    # Declaring index, using most of the default parameters from https://github.com/nmslib/hnswlib
    p = hnswlib.Index(space = 'cosine', dim = ref_cell_embeddings.shape[1]) # possible options are l2, cosine or ip
    p.init_index(max_elements = ref_cell_embeddings.shape[0], ef_construction = 200, M = 16)

    # Element insertion (can be called several times):
    p.add_items(ref_cell_embeddings, ids = np.arange(ref_cell_embeddings.shape[0]))

    # Controlling the recall by setting ef:
    p.set_ef(50) # ef should always be > k

    # Query dataset, k - number of closest elements (returns 2 numpy arrays)
    labels, distances = p.knn_query(test_embed, k = k)

idx_list=[i for i in range(test_embed.shape[0])]
gt_list = []
pred_list = []

for k in idx_list:
    gt = y_test[k]
    if hnswlib_imported:
        idx = labels[k]
    else:
        idx, sim = get_similar_vectors(test_embed[k][np.newaxis, ...], ref_cell_embeddings)

    # Neighbors' labels
    neighbor_labels = y_train[idx]
    neighbors_list_gpt_v1.append(neighbor_labels)

    # Majority vote (works with strings too)
    pred = Counter(neighbor_labels).most_common(1)[0][0]

    gt_list.append(gt)
    pred_list.append(pred)

acc = sklearn.metrics.accuracy_score(gt_list, pred_list)
precision, recall, f1, _ = sklearn.metrics.precision_recall_fscore_support(
    gt_list, pred_list, average='macro'
)

classes = np.unique(y_train)  # all possible cell types

# One-hot encode true labels
y_true_bin = label_binarize(gt_list, classes=classes)

# Get neighbor-weighted probabilities as scores (recommended for PR-AUC)
proba_list = []
for neigh in neighbors_list_gpt_v1:
    counts = Counter(neigh)
    probs = np.array([counts.get(c, 0) / len(neigh) for c in classes])
    proba_list.append(probs)

y_score = np.array(proba_list)

# Compute macro-average PR-AUC
pr_auc = average_precision_score(y_true_bin, y_score, average='macro')
print(f"Macro PR-AUC: {pr_auc:.3f}")

auc = roc_auc_score(y_true_bin, y_score, average='macro')
print("ROC-AUC:", auc)

print(f"Accuracy: {acc:.3f} Precision: {precision:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}")